# CS781 Phase 7 — A3 LoRA

This is my **a3_lora** run for Prior Case Retrieval.

A3 local RoPE/GQA model with additive low-rank Q/V updates and a retrieval head.

**Steps:** install packages → get Phase 1 checkpoint/tokenizer → get official Phase 7 data → train → check dev Micro-F1@7 → rank test candidates → make `prediction.json` and `submission.zip` → private HF backup.

- Run this on **Kaggle**, with **GPU and Internet enabled**.
- Kaggle Secrets must contain `HF` with permission to read the Phase 1 checkpoint repos.
- Keep the dataset splits separate. Do not use test labels.
- All important training settings are in Cell 2. I can rerun a failed cell; `last.pt` is saved during training.
- A script copy is included inside this notebook, so no extra Python file is needed to start.


## 1. Install libraries

In [ ]:
# Using the packages needed for my own model classes. No PEFT or torchao needed.
%pip -q install 'transformers==4.57.1' 'huggingface_hub>=0.34,<1' 'tokenizers>=0.22,<0.23' safetensors scikit-learn

## 2. Settings

In [ ]:
import os, random, json, shutil
from pathlib import Path
import numpy as np
import torch

# My experiment
ARCH = 'a3'
METHOD = 'lora'
SLUG = 'a3_lora'
OUT = Path('/kaggle/working/phase7_'+SLUG)
OUT.mkdir(parents=True, exist_ok=True)

# Phase 1 repo names are in phase7_setup.py below.
# Set one of these only if I already have files saved locally.
LOCAL_PHASE1_DIR = ''
LOCAL_PHASE7_DATA_DIR = ''

SEED = 261110044
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Longer documents: first 256 and last 256 tokens, combined as one case vector.
MAX_TOKENS = 256
CHUNKS_PER_CASE = 2
EMBED_BATCH_SIZE = 6
HYBRID_ALPHA = 0.35     # dense similarity + TF-IDF lexical overlap

# Training: change these values before a new experiment, not halfway through a saved run.
EPOCHS = 3
LR = 1e-4
ACCUM_STEPS = 4
SAVE_STEPS = 40
MAX_TRAIN_QUERIES = None     # None = use all training queries
UPLOAD_EACH_EPOCH = True    # private HF backup after each full epoch
HF_BACKUP_REPO = 'cs781-phase7-'+SLUG.replace('_','-')

print('Experiment:',SLUG)
print('Device:',DEVICE,'CUDA:',torch.cuda.is_available())
print('Output folder:',OUT)


## 3. Original Phase 1 transformer blocks and retrieval head

In [ ]:
%%writefile /kaggle/working/phase7_backbones.py
"""Original Phase 1 transformer layers, reused for Phase 7.

Only the task-specific language-model output is dropped: retrieval needs
representations, not next-token logits. Checkpoint keys are checked strictly.
"""
import re
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from types import SimpleNamespace

# Phase 1 forward methods returned these HF objects. No generation is used here.
def CausalLMOutput(**kwargs):
    return SimpleNamespace(**kwargs)

def Seq2SeqLMOutput(**kwargs):
    return SimpleNamespace(**kwargs)

# ---- A1 original decoder block ----
class DecoderBlock(nn.Module):
    """
    One Transformer decoder block.

    Structure:
        LayerNorm
        -> Causal Multi-Head Self Attention
        -> Residual Connection
        -> LayerNorm
        -> Feed Forward Network
        -> Residual Connection
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        dropout=0.1
    ):
        super().__init__()

        # Normalize before self-attention.
        self.norm1 = nn.LayerNorm(
            d_model
        )

        # Multi-head self-attention.
        #
        # Query, Key and Value all come from
        # the same token sequence.
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True
        )

        # Normalize before feed-forward network.
        self.norm2 = nn.LayerNorm(
            d_model
        )

        # Transformer feed-forward network.
        self.feed_forward = nn.Sequential(
            nn.Linear(
                d_model,
                d_ff
            ),
            nn.GELU(),
            nn.Linear(
                d_ff,
                d_model
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        causal_mask
    ):

        # ----------------------------------------------------
        # 1. Causal self-attention
        # ----------------------------------------------------

        z = self.norm1(x)

        attention_output, _ = self.attention(
            z,
            z,
            z,
            attn_mask=causal_mask,
            need_weights=False
        )

        # Residual connection.
        x = x + self.dropout(
            attention_output
        )

        # ----------------------------------------------------
        # 2. Feed-forward network
        # ----------------------------------------------------

        z = self.norm2(x)

        ff_output = self.feed_forward(
            z
        )

        # Second residual connection.
        x = x + self.dropout(
            ff_output
        )

        return x



# ---- A2 original encoder block with padding mask ----
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        # Normalize before attention, as in the original pre-LN block.
        z = self.norm1(x)
        attention_output, _ = self.self_attention(
            z, z, z, key_padding_mask=key_padding_mask, need_weights=False)
        # Add the attention output back to the input (residual connection).
        x = x + self.dropout(attention_output)
        z = self.norm2(x)
        return x + self.dropout(self.feed_forward(z))

# NER uses token features from the encoder. The old text-generation decoder is not used.
class Phase1A2Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, source_length, n_heads, n_layers, d_ff):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.source_position = nn.Embedding(source_length, d_model)
        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])

    def forward(self, input_ids, attention_mask):
        seq_len = input_ids.shape[1]
        assert seq_len <= self.source_position.num_embeddings
        # Add a learned position vector to each token embedding.
        pos = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.source_position(pos)
        # Tell attention to ignore padding positions in the batch.
        padding_mask = (attention_mask == 0)
        for block in self.encoder_blocks:
            x = block(x, key_padding_mask=padding_mask)
        return x

# ---- A3 original RMSNorm, GQA, RoPE and SwiGLU blocks ----
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Compute the RMS in fp32 for numerical stability, then cast back.
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x_norm = x * torch.rsqrt(rms + self.eps).to(x.dtype)
        return x_norm * self.weight


def rotate_half(x):
    # RoPE rotates each consecutive pair: (x0, x1) -> (-x1, x0).
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    return torch.stack((-x_odd, x_even), dim=-1).flatten(-2)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE needs an even head dimension."
        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def cos_sin(self, positions, dtype):
        # positions: [seq]
        freqs = torch.outer(positions.float(), self.inv_freq)
        cos = freqs.cos().repeat_interleave(2, dim=-1).to(dtype)
        sin = freqs.sin().repeat_interleave(2, dim=-1).to(dtype)
        return cos[None, None, :, :], sin[None, None, :, :]

    def apply_rotary(self, q, k, positions):
        cos, sin = self.cos_sin(positions, q.dtype)
        q = (q * cos) + (rotate_half(q) * sin)
        k = (k * cos) + (rotate_half(k) * sin)
        return q, k


class GroupedQueryLocalAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        window_size,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = dim // num_heads
        self.groups = num_heads // num_kv_heads
        self.window_size = window_size
        self.chunk_size = chunk_size
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, dim, bias=False)

        self.rope = RotaryEmbedding(self.head_dim)

    def _repeat_kv(self, x):
        # [B, kv_heads, T, D] -> [B, query_heads, T, D]
        return x.repeat_interleave(self.groups, dim=1)

    def _local_attention(self, q, k, v):
        """
        Exact causal sliding-window attention.

        We process a few query positions at a time.  Each query chunk only
        materializes the K/V region that can actually be attended to, instead
        of creating a full 2048 x 2048 attention matrix.
        """
        B, H, T, D = q.shape
        outputs = []

        k_rep = self._repeat_kv(k)
        v_rep = self._repeat_kv(v)

        for q_start in range(0, T, self.chunk_size):
            q_end = min(q_start + self.chunk_size, T)

            # Earliest key any query in this chunk could legally use.
            k_start = max(0, q_start - self.window_size + 1)
            k_end = q_end

            q_chunk = q[:, :, q_start:q_end, :]
            k_chunk = k_rep[:, :, k_start:k_end, :]
            v_chunk = v_rep[:, :, k_start:k_end, :]

            q_pos = torch.arange(q_start, q_end, device=q.device)
            k_pos = torch.arange(k_start, k_end, device=q.device)

            # True = allowed attention position.
            causal = k_pos[None, :] <= q_pos[:, None]
            within_window = k_pos[None, :] >= (
                q_pos[:, None] - self.window_size + 1
            )
            attn_mask = (causal & within_window)[None, None, :, :]

            out = F.scaled_dot_product_attention(
                q_chunk,
                k_chunk,
                v_chunk,
                attn_mask=attn_mask,
                dropout_p=self.dropout if self.training else 0.0,
                is_causal=False,
            )
            outputs.append(out)

        return torch.cat(outputs, dim=2)

    def forward(
        self,
        x,
        past_kv=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T, _ = x.shape

        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=x.device,
        )
        q, k = self.rope.apply_rotary(q, k, positions)

        if past_kv is None:
            # Training/prefill path: causal local attention over this sequence.
            attn_out = self._local_attention(q, k, v)
            k_all, v_all = k, v
        else:
            # Generation path.  We decode one token at a time after prefill.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv
            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            # Keep only the most recent local window in the KV cache.
            k_all = k_all[:, :, -self.window_size:, :]
            v_all = v_all[:, :, -self.window_size:, :]

            k_rep = self._repeat_kv(k_all)
            v_rep = self._repeat_kv(v_all)

            # Every cached token is in the past (or the current token), so no
            # extra causal mask is necessary for this one-token decode step.
            attn_out = F.scaled_dot_product_attention(
                q,
                k_rep,
                v_rep,
                dropout_p=0.0,
                is_causal=False,
            )

        new_kv = None
        if use_cache:
            new_kv = (
                k_all[:, :, -self.window_size:, :],
                v_all[:, :, -self.window_size:, :],
            )

        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.dim)
        return self.o_proj(attn_out), new_kv


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.down_proj = nn.Linear(hidden_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.dropout(self.down_proj(x))


class ModernTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        num_kv_heads,
        ffn_dim,
        window_size,
        chunk_size,
        dropout,
    ):
        super().__init__()
        self.attn_norm = RMSNorm(dim)
        self.attn = GroupedQueryLocalAttention(
            dim=dim,
            num_heads=num_heads,
            num_kv_heads=num_kv_heads,
            window_size=window_size,
            chunk_size=chunk_size,
            dropout=dropout,
        )
        self.ffn_norm = RMSNorm(dim)
        self.ffn = SwiGLU(dim, ffn_dim, dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, position_offset=0):
        attn_out, new_kv = self.attn(
            self.attn_norm(x),
            past_kv=past_kv,
            use_cache=use_cache,
            position_offset=position_offset,
        )
        x = x + self.resid_dropout(attn_out)
        x = x + self.ffn(self.ffn_norm(x))
        return x, new_kv


class ModernLocalTransformerLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim=600,
        num_layers=8,
        num_heads=10,
        num_kv_heads=5,
        ffn_dim=1600,
        window_size=512,
        chunk_size=256,
        dropout=0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        self.num_layers = num_layers

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            ModernTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                num_kv_heads=num_kv_heads,
                ffn_dim=ffn_dim,
                window_size=window_size,
                chunk_size=chunk_size,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = RMSNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Weight tying saves parameters and is standard for causal LMs.
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        x = self.embed_dropout(self.token_embedding(input_ids))
        new_caches = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, new_kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                position_offset=position_offset,
            )
            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)
        return x, new_caches

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        x, _ = self._run_hidden(input_ids)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            # Position t predicts token t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(loss=loss, logits=logits)

    @torch.inference_mode()
    def prefill(self, input_ids):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )
        return self.lm_head(x), caches

    @torch.inference_mode()
    def decode_step(self, input_ids, past_key_values, position_offset):
        x, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )
        return self.lm_head(x[:, -1, :]), caches


# ---- A4 original multi-scale and dilated attention blocks ----
def build_pattern_mask(context_length, pattern_type, pattern_value=None):
    """
    Build a boolean causal attention mask of shape [context_length, context_length].

    PyTorch SDPA interprets True as "this query may attend to this key".

    Example for dilation=2 at query position t:
        allowed keys = t, t-2, t-4, t-6, ...
    """
    query_pos = torch.arange(context_length).unsqueeze(1)  # [T, 1]
    key_pos = torch.arange(context_length).unsqueeze(0)    # [1, T]

    distance = query_pos - key_pos
    causal = distance >= 0

    if pattern_type == "global":
        allowed = causal

    elif pattern_type == "local":
        window = int(pattern_value)
        assert window >= 1
        allowed = causal & (distance < window)

    elif pattern_type == "dilated":
        dilation = int(pattern_value)
        assert dilation >= 1
        allowed = causal & ((distance % dilation) == 0)

    else:
        raise ValueError(f"Unknown attention pattern: {pattern_type}")

    return allowed


class PatternCausalSelfAttention(nn.Module):
    """Causal self-attention with a chosen local/dilated/global pattern."""

    def __init__(
        self,
        dim,
        num_heads,
        context_length,
        pattern_type,
        pattern_value=None,
        dropout=0.1,
    ):
        super().__init__()
        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.context_length = context_length
        self.pattern_type = pattern_type
        self.pattern_value = pattern_value
        self.dropout = dropout

        self.q_proj = nn.Linear(dim, dim, bias=False)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)

        # The mask is fixed by the architecture, so build it only once.
        # persistent=False means this large mask is NOT written into checkpoints.
        pattern_mask = build_pattern_mask(
            context_length=context_length,
            pattern_type=pattern_type,
            pattern_value=pattern_value,
        )
        self.register_buffer(
            "pattern_mask",
            pattern_mask,
            persistent=False,
        )

    def forward(self, x, past_kv=None, use_cache=False):
        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        if past_kv is None:
            # Training / prompt prefill.
            k_all = k
            v_all = v
            key_length = T
            query_start = 0

        else:
            # Incremental generation: append the new token's K/V to the cache.
            assert T == 1, "Cached decoding expects one new token at a time."
            past_k, past_v = past_kv

            k_all = torch.cat([past_k, k], dim=2)
            v_all = torch.cat([past_v, v], dim=2)

            key_length = k_all.size(2)
            query_start = key_length - T

        if key_length > self.context_length:
            raise ValueError(
                f"Attention cache length {key_length} exceeds "
                f"context length {self.context_length}."
            )

        # Slice exactly the query rows and key columns needed in this call.
        # Shape becomes [1, 1, query_length, key_length] and broadcasts over
        # batch and attention heads.
        attn_mask = self.pattern_mask[
            query_start:query_start + T,
            :key_length,
        ][None, None, :, :]

        attn_out = F.scaled_dot_product_attention(
            q,
            k_all,
            v_all,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,  # causality is already encoded in attn_mask
        )

        new_kv = (k_all, v_all) if use_cache else None

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, self.dim)
        )

        return self.o_proj(attn_out), new_kv


class MultiScaleTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        ffn_dim,
        context_length,
        pattern_type,
        pattern_value=None,
        dropout=0.1,
    ):
        super().__init__()

        # Pre-LayerNorm Transformer block.
        self.norm1 = nn.LayerNorm(dim)
        self.attn = PatternCausalSelfAttention(
            dim=dim,
            num_heads=num_heads,
            context_length=context_length,
            pattern_type=pattern_type,
            pattern_value=pattern_value,
            dropout=dropout,
        )

        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, dim),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False):
        attn_out, new_kv = self.attn(
            self.norm1(x),
            past_kv=past_kv,
            use_cache=use_cache,
        )

        # Residual connection around attention.
        x = x + self.dropout(attn_out)

        # Residual connection around feed-forward network.
        x = x + self.dropout(self.ffn(self.norm2(x)))

        return x, new_kv


class MultiScaleDilatedLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_length=2048,
        dim=600,
        num_heads=10,
        ffn_dim=1600,
        attention_patterns=None,
        dropout=0.1,
    ):
        super().__init__()

        if attention_patterns is None:
            raise ValueError("attention_patterns must be provided.")

        self.vocab_size = vocab_size
        self.context_length = context_length
        self.dim = dim
        self.num_heads = num_heads
        self.ffn_dim = ffn_dim
        self.attention_patterns = attention_patterns

        self.token_embedding = nn.Embedding(vocab_size, dim)
        self.position_embedding = nn.Embedding(context_length, dim)
        self.embed_dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            MultiScaleTransformerBlock(
                dim=dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                context_length=context_length,
                pattern_type=pattern["type"],
                pattern_value=pattern.get("value"),
                dropout=dropout,
            )
            for pattern in attention_patterns
        ])

        self.final_norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

        # Initialize first, then tie token-embedding and output weights.
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_embedding.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _run_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_offset=0,
    ):
        B, T = input_ids.shape

        if position_offset + T > self.context_length:
            raise ValueError(
                f"Sequence position {position_offset + T} exceeds "
                f"context length {self.context_length}."
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=input_ids.device,
        )

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)[None, :, :]
        )
        x = self.embed_dropout(x)

        if past_key_values is None:
            layer_past = [None] * len(self.blocks)
        else:
            if len(past_key_values) != len(self.blocks):
                raise ValueError("Wrong number of layer caches.")
            layer_past = past_key_values

        new_caches = []

        for block, past_kv in zip(self.blocks, layer_past):
            x, new_kv = block(
                x,
                past_kv=past_kv,
                use_cache=use_cache,
            )

            if use_cache:
                new_caches.append(new_kv)

        x = self.final_norm(x)

        return x, (new_caches if use_cache else None)

    def forward(self, input_ids, labels=None, attention_mask=None, **kwargs):
        # Training examples contain full fixed-length token blocks, so no
        # padding mask is needed here. The architectural causal/sparse mask is
        # handled inside each attention layer.
        hidden, _ = self._run_hidden(input_ids)
        logits = self.lm_head(hidden)

        loss = None

        if labels is not None:
            # Next-token language modelling:
            # position t predicts label at position t+1.
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
            )

        return CausalLMOutput(
            loss=loss,
            logits=logits,
        )

    @torch.inference_mode()
    def prefill(self, input_ids):
        """Process the full prompt once and create K/V caches for all layers."""
        hidden, caches = self._run_hidden(
            input_ids,
            past_key_values=None,
            use_cache=True,
            position_offset=0,
        )

        return self.lm_head(hidden), caches

    @torch.inference_mode()
    def decode_step(
        self,
        input_ids,
        past_key_values,
        position_offset,
    ):
        """Process one newly generated token using the existing K/V caches."""
        hidden, caches = self._run_hidden(
            input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            position_offset=position_offset,
        )

        logits = self.lm_head(hidden[:, -1, :])
        return logits, caches


# ------------------------------------------------------------
# Create Architecture 4.
# set_seed(SEED) was already executed in the configuration cell,
# BEFORE this model is initialized.
# ------------------------------------------------------------

class Phase1A1Encoder(nn.Module):
    # Same A1 token/position embeddings, decoder blocks, and final norm.
    # Vocabulary prediction layer is unnecessary for retrieval.
    def __init__(self, vocab_size, context_length, d_model, n_heads, n_layers, d_ff, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(context_length, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.context_length = context_length

    def forward(self, ids, attention_mask=None, adapters=None):
        n = ids.size(1)
        if n > self.context_length:
            raise ValueError('A1 document chunk exceeds the Phase 1 context limit')
        pos = torch.arange(n,device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(pos)[None,:,:]
        mask = torch.triu(torch.ones(n,n,device=ids.device,dtype=torch.bool),diagonal=1)
        for i,block in enumerate(self.blocks):
            x = block(x, mask)
            if adapters is not None:
                x = x + adapters[i](x)
        return self.final_norm(x)


class RetrievalModel(nn.Module):
    # Takes token chunks and returns one normalized vector per chunk.
    def __init__(self, base, arch, dim, method, rank=8, proj_dim=256):
        super().__init__()
        self.base = base
        self.arch = arch
        self.method = method
        self.adapters = nn.ModuleList()
        self.projection = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim,proj_dim), nn.Tanh())
        for p in self.base.parameters():
            p.requires_grad_(method in ('full','teacher','student'))
        if method == 'partial':
            if arch != 'a2':
                raise ValueError('Partial variant is implemented for A2')
            for block in self.base.encoder_blocks[-2:]:
                for p in block.parameters(): p.requires_grad_(True)
        if method == 'adapters':
            if arch != 'a1': raise ValueError('Adapter experiment is for A1')
            self.adapters = nn.ModuleList([BottleneckAdapter(dim) for _ in self.base.blocks])
        if method == 'lora':
            if arch != 'a3': raise ValueError('LoRA experiment is for A3')
            for block in self.base.blocks:
                block.attn.q_proj = LoRALinear(block.attn.q_proj,rank)
                block.attn.v_proj = LoRALinear(block.attn.v_proj,rank)
        if method == 'lorma':
            if arch != 'a4': raise ValueError('LORMA experiment is for A4')
            for block in self.base.blocks:
                block.attn.q_proj = LoRMALinear(block.attn.q_proj,rank)
                block.attn.v_proj = LoRMALinear(block.attn.v_proj,rank)
                block.ffn[0] = LoRMALinear(block.ffn[0],rank)
                block.ffn[-1] = LoRMALinear(block.ffn[-1],rank)

    def forward(self, input_ids, attention_mask):
        if self.arch == 'a1':
            x = self.base(input_ids, attention_mask, self.adapters if len(self.adapters) else None)
        elif self.arch == 'a2':
            x = self.base(input_ids, attention_mask)
        else:
            x, _ = self.base._run_hidden(input_ids)
        # Trailing padding must not contribute to the mean.
        mask = attention_mask.to(x.dtype).unsqueeze(-1)
        pooled = (x*mask).sum(dim=1)/mask.sum(dim=1).clamp(min=1)
        return F.normalize(self.projection(pooled).float(),dim=-1)


class BottleneckAdapter(nn.Module):
    def __init__(self, dim, bottleneck=32):
        super().__init__()
        self.down = nn.Linear(dim,bottleneck)
        self.up = nn.Linear(bottleneck,dim)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self,x):
        return self.up(F.gelu(self.down(x)))


class LoRALinear(nn.Module):
    # LoRA: W x + B A x (original W stays frozen).
    def __init__(self, original, rank=8, alpha=16):
        super().__init__()
        self.original = original
        self.original.requires_grad_(False)
        self.down = nn.Linear(original.in_features,rank,bias=False)
        self.up = nn.Linear(rank,original.out_features,bias=False)
        nn.init.kaiming_uniform_(self.down.weight, a=5**0.5)
        nn.init.zeros_(self.up.weight)
        self.scale=alpha/rank
    def forward(self,x):
        return self.original(x)+self.up(self.down(x))*self.scale


class LoRMALinear(nn.Module):
    # LORMA: (I + BA) (Wx). Different from the usual LoRA update.
    def __init__(self, original, rank=8, alpha=16):
        super().__init__()
        self.original=original
        self.original.requires_grad_(False)
        self.down=nn.Linear(original.out_features,rank,bias=False)
        self.up=nn.Linear(rank,original.out_features,bias=False)
        nn.init.kaiming_uniform_(self.down.weight, a=5**0.5)
        nn.init.zeros_(self.up.weight)
        self.scale=alpha/rank
    def forward(self,x):
        y=self.original(x)
        return y+self.up(self.down(y))*self.scale


def _layer_count(state,prefix):
    found=[int(m.group(1)) for k in state if (m:=re.match(r'^'+prefix+r'\.(\d+)\.',k))]
    if not found: raise ValueError('Cannot locate '+prefix+' in Phase 1 weights')
    return max(found)+1


def _check_and_load(model, original):
    # Strictly load ALL encoder/retrieval-backbone tensors.
    expected=model.state_dict()
    subset={k:v for k,v in original.items() if k in expected}
    # Safetensors may store only one name for tied embedding / LM head.
    for a,b in [('lm_head.weight','token_embedding.weight'),('token_embedding.weight','lm_head.weight')]:
        if a in expected and a not in subset and b in subset:
            subset[a]=subset[b]
    missing=sorted(set(expected)-set(subset))
    wrong=[k for k,v in subset.items() if tuple(v.shape)!=tuple(expected[k].shape)]
    if missing or wrong:
        raise ValueError(f'Phase 1 checkpoint mismatch. Missing={missing[:5]}, wrong shapes={wrong[:5]}')
    model.load_state_dict(subset,strict=True)
    return len(subset)


def build_backbone(arch, state, cfg=None, student_layers=None):
    # Dimensions come from the actual saved weights, not notebook assumptions.
    cfg=cfg or {}
    vocab,dim=state['token_embedding.weight'].shape
    if arch=='a1':
        layers=_layer_count(state,'blocks')
        context=state['position_embedding.weight'].shape[0]
        dff=state['blocks.0.feed_forward.0.weight'].shape[0]
        if dim!=640 and 'n_heads' not in cfg:
            raise ValueError('Unknown A1 head count. Supply Phase 1 architecture_config.json')
        base=Phase1A1Encoder(vocab,context,dim,int(cfg.get('n_heads',10)),layers,dff)
    elif arch=='a2':
        layers=_layer_count(state,'encoder_blocks')
        context=state['source_position.weight'].shape[0]
        dff=state['encoder_blocks.0.feed_forward.0.weight'].shape[0]
        if dim!=600 and 'n_heads' not in cfg:
            raise ValueError('Unknown A2 head count. Supply Phase 1 architecture_config.json')
        base=Phase1A2Encoder(vocab,dim,context,int(cfg.get('n_heads',15)),layers,dff)
    elif arch=='a3':
        layers=_layer_count(state,'blocks')
        dff=state['blocks.0.ffn.gate_proj.weight'].shape[0]
        # RoPE/KV head count cannot be inferred from the tensor shape.
        for key in ('num_heads','num_kv_heads','window_size','chunk_size'):
            if key not in cfg: raise ValueError(f'A3 config is missing {key}. Do not guess.')
        if student_layers is not None: layers=int(student_layers)
        base=ModernLocalTransformerLM(vocab,dim,layers,int(cfg['num_heads']),
             int(cfg['num_kv_heads']),dff,int(cfg['window_size']),int(cfg['chunk_size']),0.1)
    elif arch=='a4':
        patterns=cfg.get('attention_patterns')
        if not patterns or 'num_heads' not in cfg:
            raise ValueError('A4 requires original attention_patterns and num_heads from saved config')
        layers=_layer_count(state,'blocks')
        if len(patterns)!=layers:
            raise ValueError('A4 attention patterns disagree with Phase 1 layers')
        if student_layers is not None: patterns=patterns[:student_layers]
        dff=state['blocks.0.ffn.0.weight'].shape[0]
        context=state['position_embedding.weight'].shape[0]
        base=MultiScaleDilatedLM(vocab,context,dim,int(cfg['num_heads']),dff,patterns,0.1)
    else:
        raise ValueError('Unknown architecture: '+arch)
    count=_check_and_load(base,state)
    return base, dim, vocab, count


## 4. Retrieval, loss, local metrics and JSON checks

In [ ]:
%%writefile /kaggle/working/phase7_core.py
"""Shared Phase 7 training, retrieval, evaluation, and submission checks.

This file is included inside each Kaggle notebook. Only official data is used.
"""
import os
import re
import io
import json
import time
import random
import hashlib
import zipfile
from pathlib import Path
from functools import lru_cache
from contextlib import nullcontext

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.feature_extraction.text import TfidfVectorizer


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def read_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_records(path, kind):
    data = read_json(path)
    # Supports JSON objects keyed by official IDs and JSON lists with an id field.
    while isinstance(data, dict) and len(data) == 1:
        k = next(iter(data))
        if k in ('data', 'queries', 'candidates', 'documents', 'items', 'examples'):
            data = data[k]
        else:
            break
    if isinstance(data, list):
        result = {}
        for entry in data:
            if not isinstance(entry, dict):
                raise ValueError(f'{kind}: expected an object for each record')
            raw = next((entry[k] for k in ('id','query_id','case_id','candidate_id') if k in entry), None)
            if not isinstance(raw, str):
                raise ValueError(f'{kind}: case IDs must be strings so leading zeros remain unchanged')
            if raw in result: raise ValueError(f'{kind}: repeated id {raw}')
            result[raw] = entry
        return result
    if isinstance(data, dict):
        if not all(isinstance(k, str) for k in data):
            raise ValueError(f'{kind}: keys must be strings')
        return {k:(v if isinstance(v,dict) else {'text':v}) for k,v in data.items()}
    raise ValueError(f'{kind}: expected list or dictionary in {path}')


def sentence_text(obj):
    if isinstance(obj, str): return obj
    if isinstance(obj, list): return ' '.join(sentence_text(x) for x in obj)
    if isinstance(obj, dict):
        for k in ('text','sentence','content','document','case_text'):
            if k in obj: return sentence_text(obj[k])
    return ''


def case_text(entry):
    return sentence_text(entry)


def gold_ids(entry):
    if not isinstance(entry,dict) or 'relevant_candidates' not in entry:
        raise ValueError('A training/dev query does not have relevant_candidates')
    result=entry['relevant_candidates']
    if isinstance(result,dict):
        result=[k for k,v in result.items() if v]
    if not isinstance(result,list) or not all(isinstance(x,str) for x in result):
        raise ValueError('relevant_candidates must be a list of string case IDs')
    return list(dict.fromkeys(result))


def check_splits(train_q,train_c,dev_q,dev_c,test_q,test_c):
    for name,qs,cs in [('train',train_q,train_c),('dev',dev_q,dev_c),('test',test_q,test_c)]:
        if len(cs)<7: raise ValueError(f'{name} has fewer than 7 candidates')
        if not qs: raise ValueError(f'{name} has no queries')
        if any(not case_text(v).strip() for v in list(qs.values())[:10]):
            raise ValueError(f'{name}: query text is missing')
        if any(not case_text(v).strip() for v in list(cs.values())[:10]):
            raise ValueError(f'{name}: candidate text is missing')
        if name!='test':
            absent={pid for q in qs.values() for pid in gold_ids(q) if pid not in cs}
            if absent: raise ValueError(f'{name}: gold IDs outside candidate pool: {sorted(absent)[:5]}')
    print('Official splits:',{s:(len(q),len(c)) for s,q,c in
          [('train',train_q,train_c),('dev',dev_q,dev_c),('test',test_q,test_c)]})


class CaseTokens:
    def __init__(self, tokenizer, max_tokens=256, n_chunks=2):
        self.tokenizer=tokenizer
        self.max_tokens=int(max_tokens)
        self.n_chunks=int(n_chunks)
        self.pad_id=tokenizer.pad_token_id
        if self.pad_id is None:
            self.pad_id=tokenizer.eos_token_id
        if self.pad_id is None: self.pad_id=0
        if self.max_tokens < 8: raise ValueError('max_tokens is too small')

    @lru_cache(maxsize=50000)
    def chunks(self, text):
        # Head + tail usually cover facts and decision. Avoid encoding huge documents.
        # Keep the middle when we can afford a third chunk.
        if not text.strip(): text='[empty document]'
        if len(text) > 24000:
            text=text[:10000]+'\n'+text[len(text)//2-2000:len(text)//2+2000]+'\n'+text[-10000:]
        ids=self.tokenizer.encode(text,add_special_tokens=False)
        if not ids: ids=[self.pad_id]
        m=self.max_tokens
        if len(ids)<=m: return (tuple(ids),)
        if self.n_chunks==1:
            n=m//2
            return (tuple(ids[:n]+ids[-(m-n):]),)
        if self.n_chunks==2:
            return (tuple(ids[:m]),tuple(ids[-m:]))
        indices=np.linspace(0,max(0,len(ids)-m),self.n_chunks,dtype=int)
        return tuple(tuple(ids[i:i+m]) for i in indices)

    def pack(self, texts, device):
        # [cases * chunks, token positions], and a mask for missing chunks.
        allchunks=[self.chunks(str(text)) for text in texts]
        n=self.n_chunks
        packed=[]; valid=[]
        for chunks in allchunks:
            seq=list(chunks[:n]); ok=[1]*len(seq)
            while len(seq)<n:
                seq.append((self.pad_id,));ok.append(0)
            packed.extend(seq)
            valid.append(ok)
        length=max(len(s) for s in packed)
        tokens=torch.full((len(packed),length),self.pad_id,dtype=torch.long)
        mask=torch.zeros((len(packed),length),dtype=torch.long)
        for i,s in enumerate(packed):
            tokens[i,:len(s)]=torch.tensor(s,dtype=torch.long)
            mask[i,:len(s)]=1
        return tokens.to(device),mask.to(device),torch.tensor(valid,dtype=torch.float32,device=device)


def embed_cases(model, texts, case_tokens, device):
    # The same model is used for a query and a candidate (bi-encoder).
    ids,mask,valid=case_tokens.pack(texts,device)
    embs=model(ids,mask).reshape(len(texts),case_tokens.n_chunks,-1)
    weighted=(embs*valid.unsqueeze(-1)).sum(1)/valid.sum(1,keepdim=True).clamp(min=1)
    return F.normalize(weighted.float(),dim=-1)


def vectors_for_records(model, records, case_tokens, device, batch_size=8):
    model.eval()
    names=sorted(records)
    batches=[]
    with torch.no_grad():
        for start in range(0,len(names),batch_size):
            subset=names[start:start+batch_size]
            text=[case_text(records[k]) for k in subset]
            with mixed_precision(device):
                vec=embed_cases(model,text,case_tokens,device)
            batches.append(vec.float().cpu().numpy())
    return names,np.concatenate(batches,axis=0)


def mixed_precision(device):
    if device.type!='cuda': return nullcontext()
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast('cuda',dtype=dtype)


def build_lexical(queries,candidates):
    # Fit only to the pool of the same split. No external legal corpus is used.
    cand_ids=sorted(candidates)
    texts=[case_text(candidates[k])[:20000] for k in cand_ids]
    vectorizer=TfidfVectorizer(max_features=45000,ngram_range=(1,2),sublinear_tf=True,dtype=np.float32)
    cand_matrix=vectorizer.fit_transform(texts)
    query_matrix=vectorizer.transform([case_text(queries[k])[:20000] for k in sorted(queries)])
    return cand_ids,cand_matrix,sorted(queries),query_matrix


def standardize(scores):
    scores=np.asarray(scores,dtype=np.float32)
    return (scores-scores.mean())/max(float(scores.std()),1e-6)


def validate_predictions(queries,candidates,predictions):
    if not isinstance(predictions,dict) or set(queries)!=set(predictions):
        missing=set(queries)-set(predictions); extra=set(predictions)-set(queries)
        raise ValueError(f'Prediction query ID mismatch. Missing {sorted(missing)[:4]} extra {sorted(extra)[:4]}')
    allowed=set(candidates)
    for q,ranked in predictions.items():
        if not isinstance(ranked,list) or len(ranked)!=7:
            raise ValueError(f'{q}: need exactly 7 candidate IDs')
        if not all(isinstance(x,str) for x in ranked):
            raise ValueError(f'{q}: case IDs must be strings')
        if len(set(ranked))!=7 or not set(ranked)<=allowed:
            raise ValueError(f'{q}: duplicate or out-of-pool candidate IDs')


def metric_at7(queries,predictions):
    tp=gold=0
    for q,entry in queries.items():
        goldset=set(gold_ids(entry))
        gold+=len(goldset)
        tp+=len(set(predictions[q]) & goldset)
    p=tp/(7*len(queries)) if queries else 0.0
    r=tp/gold if gold else 0.0
    f=2*p*r/(p+r) if p+r else 0.0
    return {'micro_precision_at7':p,'micro_recall_at7':r,'micro_f1_at7':f,
            'correct':tp,'gold_count':gold,'predicted_count':7*len(queries)}


def rank_split(model,queries,candidates,case_tokens,device,alpha=0.35,batch_size=8):
    # Dense legal vectors + word overlap. Fallback lexical signal helps statutes/citations.
    ids,candidate_vec=vectors_for_records(model,candidates,case_tokens,device,batch_size)
    q_ids,query_vec=vectors_for_records(model,queries,case_tokens,device,batch_size)
    lex_ids,lex_candidates,lex_q_ids,lex_queries=build_lexical(queries,candidates)
    if ids!=lex_ids or q_ids!=lex_q_ids:
        raise AssertionError('Dense and lexical case IDs are misaligned')
    predictions={}
    for i,q in enumerate(q_ids):
        dense=standardize(candidate_vec@query_vec[i])
        sparse=standardize((lex_candidates@lex_queries[i].T).toarray().reshape(-1))
        combined=(1-alpha)*dense+alpha*sparse
        top=np.argsort(-combined,kind='stable')[:7]
        predictions[q]=[ids[int(j)] for j in top]
    validate_predictions(queries,candidates,predictions)
    return predictions


def make_hard_negative_map(queries,candidates,top=30):
    cand_ids,c_mat,q_ids,q_mat=build_lexical(queries,candidates)
    output={}
    for i,q in enumerate(q_ids):
        s=(c_mat@q_mat[i].T).toarray().reshape(-1)
        # Stable ordering also avoids different sampling across runs.
        ix=np.argsort(-s,kind='stable')
        positives=set(gold_ids(queries[q]))
        output[q]=[cand_ids[j] for j in ix if cand_ids[j] not in positives][:top]
    return output


def make_triples(queries,candidates,hard_map,seed,epoch,max_queries=None):
    rng=random.Random(seed+epoch)
    keys=sorted(q for q in queries if gold_ids(queries[q]))
    rng.shuffle(keys)
    if max_queries is not None: keys=keys[:int(max_queries)]
    candidate_ids=sorted(candidates)
    triples=[]
    for q in keys:
        relevant=gold_ids(queries[q]); pos=rng.choice(relevant)
        not_gold=set(relevant)
        hard=hard_map.get(q,[])
        if hard and rng.random()<0.8:
            neg=rng.choice(hard)
        else:
            neg=rng.choice(candidate_ids)
            for _ in range(100):
                if neg not in not_gold: break
                neg=rng.choice(candidate_ids)
            else:
                neg=next((k for k in candidate_ids if k not in not_gold),None)
        if neg is None or neg in not_gold: continue
        triples.append((q,pos,neg))
    if not triples: raise ValueError('No valid positive/negative training triples')
    return triples


def save_checkpoint(path,model,optimizer,epoch,next_step,best,scaler=None):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.tmp')
    state={'model':model.state_dict(),'optimizer':optimizer.state_dict(),
           'scaler':scaler.state_dict() if scaler is not None else {},
           'epoch':epoch,'next_step':next_step,'best':best}
    torch.save(state,tmp)
    os.replace(tmp,path)


def save_best(path,model,score):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix('.tmp')
    torch.save({'model':model.state_dict(),'score':float(score)},tmp)
    os.replace(tmp,path)


def load_best(model,path):
    path=Path(path)
    if not path.exists():raise FileNotFoundError(f'No trained checkpoint found: {path}')
    state=torch.load(path,map_location='cpu',weights_only=True)
    model.load_state_dict(state['model'],strict=True)
    return float(state.get('score',0))


def train_model(model,train_q,train_c,dev_q,dev_c,case_tokens,device,folder,settings,
                teacher_vectors=None,backup_callback=None):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)
    seed=int(settings.get('seed',42)); epochs=int(settings.get('epochs',3))
    lr=float(settings.get('lr',2e-5)); accum=int(settings.get('accum_steps',4))
    alpha=float(settings.get('hybrid_alpha',0.35)); max_q=settings.get('max_train_queries')
    hard_map=make_hard_negative_map(train_q,train_c)
    params=[p for p in model.parameters() if p.requires_grad]
    if not params:raise ValueError('No trainable parameters')
    model.to(device)
    optimizer=torch.optim.AdamW(params,lr=lr,weight_decay=0.01)
    use_scaler=(device.type=='cuda' and not torch.cuda.is_bf16_supported())
    scaler=torch.amp.GradScaler('cuda',enabled=use_scaler)
    last_file=folder/'last.pt';best_file=folder/'best.pt'
    start_epoch=start_step=0;best=-1.0
    if last_file.exists():
        ckpt=torch.load(last_file,map_location='cpu',weights_only=True)
        model.load_state_dict(ckpt['model'],strict=True)
        optimizer.load_state_dict(ckpt['optimizer'])
        if ckpt.get('scaler'):scaler.load_state_dict(ckpt['scaler'])
        start_epoch=int(ckpt['epoch']);start_step=int(ckpt['next_step']);best=float(ckpt['best'])
        print(f'Resuming epoch={start_epoch+1}, step={start_step}, best={best:.5f}')
    logs=read_json(folder/'training_log.json') if (folder/'training_log.json').exists() else []
    for epoch in range(start_epoch,epochs):
        model.train()
        triples=make_triples(train_q,train_c,hard_map,seed,epoch,max_q)
        offset=start_step if epoch==start_epoch else 0
        if offset>len(triples):raise ValueError('Resume offset beyond epoch training set')
        optimizer.zero_grad(set_to_none=True)
        total_loss=0.0;step_count=0;bad_count=0
        for i in range(offset,len(triples)):
            q,pos,neg=triples[i]
            texts=[case_text(train_q[q]),case_text(train_c[pos]),case_text(train_c[neg])]
            with mixed_precision(device):
                vec=embed_cases(model,texts,case_tokens,device)
                score=torch.stack([(vec[0]*vec[1]).sum(),(vec[0]*vec[2]).sum()])
                logits=score.float().unsqueeze(0)/0.07
                gold_loss=F.cross_entropy(logits,torch.zeros(1,dtype=torch.long,device=device))
                if teacher_vectors is not None:
                    tvec_q,tvec_c=teacher_vectors
                    tq=tvec_q[q];tp=tvec_c[pos];tn=tvec_c[neg]
                    t_scores=torch.tensor([float(tq@tp),float(tq@tn)],device=device).unsqueeze(0)
                    teacher_prob=F.softmax(t_scores/0.14,dim=-1)
                    student_log=F.log_softmax(score.float().unsqueeze(0)/0.14,dim=-1)
                    kd_loss=F.kl_div(student_log,teacher_prob,reduction='batchmean')*(2.0**2)
                    loss=(0.65*gold_loss+0.35*kd_loss)/accum
                else:
                    loss=gold_loss/accum
            if not torch.isfinite(loss):
                bad_count+=1;optimizer.zero_grad(set_to_none=True)
                with (folder/'numerical_warnings.jsonl').open('a') as f:
                    f.write(json.dumps({'epoch':epoch,'step':i,'kind':'nonfinite_loss'})+'\n')
                if bad_count>=8: raise FloatingPointError('Repeated nonfinite losses; saved checkpoint remains')
                continue
            scaler.scale(loss).backward()
            total_loss+=float(loss.detach())*accum;step_count+=1
            boundary=((i+1)%accum==0 or (i+1)==len(triples))
            if not boundary:continue
            scaler.unscale_(optimizer)
            finite=all(torch.isfinite(p.grad).all().item() for p in params if p.grad is not None)
            if finite:
                torch.nn.utils.clip_grad_norm_(params,1.0)
                scaler.step(optimizer)
                scaler.update()
                bad_count=0
            else:
                bad_count+=1
                with (folder/'numerical_warnings.jsonl').open('a') as f:
                    f.write(json.dumps({'epoch':epoch,'step':i,'kind':'nonfinite_grad'})+'\n')
                if use_scaler:scaler.update(new_scale=max(float(scaler.get_scale())/2,1))
                if bad_count>=8:raise FloatingPointError('Repeated nonfinite grads; saved checkpoint remains')
            optimizer.zero_grad(set_to_none=True)
            if (i+1)%int(settings.get('save_steps',40))==0:
                save_checkpoint(last_file,model,optimizer,epoch,i+1,best,scaler)
        # Proper dev evaluation: no train labels or private test reference data.
        model.eval()
        dev_pred=rank_split(model,dev_q,dev_c,case_tokens,device,alpha,
                            int(settings.get('embed_batch_size',8)))
        metrics=metric_at7(dev_q,dev_pred)
        logs.append({'epoch':epoch+1,'train_loss':total_loss/max(step_count,1),**metrics})
        save_json(folder/'training_log.json',logs)
        save_json(folder/'dev_predictions.json',dev_pred)
        save_json(folder/'dev_metrics.json',metrics)
        print(f'Epoch {epoch+1}/{epochs}: loss={logs[-1]["train_loss"]:.4f}, dev Micro-F1@7={metrics["micro_f1_at7"]:.5f}')
        if metrics['micro_f1_at7']>best:
            best=metrics['micro_f1_at7'];save_best(best_file,model,best)
        save_checkpoint(last_file,model,optimizer,epoch+1,0,best,scaler)
        if backup_callback is not None:
            backup_callback()
        if device.type=='cuda':torch.cuda.empty_cache()
    if not best_file.exists():
        raise FileNotFoundError('Training finished without saving a valid best checkpoint')
    load_best(model,best_file)
    model.to(device).eval()
    return model


def teacher_vectors_for_train(model,queries,candidates,case_tokens,device,batch_size=8):
    cids,cvec=vectors_for_records(model,candidates,case_tokens,device,batch_size)
    qids,qvec=vectors_for_records(model,queries,case_tokens,device,batch_size)
    return ({k:v for k,v in zip(qids,qvec)}, {k:v for k,v in zip(cids,cvec)})


def prepare_submission(pred,queries,candidates,out_dir):
    out_dir=Path(out_dir);out_dir.mkdir(parents=True,exist_ok=True)
    validate_predictions(queries,candidates,pred)
    path=out_dir/'prediction.json';save_json(path,pred)
    z=out_dir/'submission.zip'
    with zipfile.ZipFile(z,'w',compression=zipfile.ZIP_DEFLATED) as f:
        f.write(path,arcname='prediction.json')
    with zipfile.ZipFile(z) as f:
        assert f.namelist()==['prediction.json']
        json.loads(f.read('prediction.json'))
    print(f'PASS: {len(pred)} queries; all IDs valid; exactly 7 per query. Submission zip: {z}')
    return path,z


## 5. HF checkpoint/data loading and backups

In [ ]:
%%writefile /kaggle/working/phase7_setup.py
"""Phase 1 checkpoint, tokenizer, and official Phase 7 data downloads."""
import os
import json
from pathlib import Path

import torch
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import load_file
from transformers import PreTrainedTokenizerFast

from phase7_backbones import build_backbone, RetrievalModel
from phase7_core import load_records, check_splits, save_json


PHASE1={
    'a1':{'repo':'P-261110044/cs781-a1-checkpoints',
          'weight':'checkpoint-6104/model.safetensors'},
    'a2':{'repo':'P-261110044/Phase1-A2',
          'weight':'cs781_phase1_a2_encoderdecoder/legal-gpt2-final/model_state.pt'},
    'a3':{'repo':'P-261110044/cs781-phase1-architecture3',
          'weight':'model_state.pt'},
    'a4':{'repo':'P-261110044/cs781-a4-checkpoints',
          'weight':'final_model/model_state.pt'},
}
DATA_REPO='Exploration-Lab/CS-781-Capstone'
DATA_PATHS={
 'train_q':'Phase_7_PCR/Train/train.json',
 'train_c':'Phase_7_PCR/Train/train_candidates.json',
 'dev_q':'Phase_7_PCR/Dev/dev.json',
 'dev_c':'Phase_7_PCR/Dev/dev_candidates.json',
 'test_q':'Phase_7_PCR/Test/test.json',
 'test_c':'Phase_7_PCR/Test/test_candidates.json',
}


def get_hf_token():
    token=os.getenv('HF_TOKEN','') or os.getenv('HF','')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret('HF')
        except Exception as err:
            print('Kaggle HF secret unavailable:',str(err)[:100])
    if not token:
        raise ValueError('Set the Kaggle secret HF (or HF_TOKEN), with access to your Phase 1 repos')
    return token


def get_remote_file(api,repo,relative,local_dir,token,repo_type='model'):
    files=set(api.list_repo_files(repo,repo_type=repo_type))
    if relative not in files:
        raise FileNotFoundError(f'{relative} is missing in {repo}. Check Phase 1 repo name/weight path.')
    return Path(hf_hub_download(repo_id=repo,filename=relative,repo_type=repo_type,
                                token=token,local_dir=str(local_dir)))


def download_phase1(arch,api,token,workspace,local_folder=None):
    spec=PHASE1[arch]
    repo=spec['repo']
    prefix=Path(spec['weight']).parent
    if str(prefix)=='.': prefix=Path('')
    if local_folder:
        remote=Path(local_folder)
        if not remote.exists(): raise FileNotFoundError(local_folder)
        files={str(x.relative_to(remote)).replace('\\','/') for x in remote.rglob('*') if x.is_file()}
        def fetch(name):return remote/name
        desc=str(remote)
    else:
        files=set(api.list_repo_files(repo,repo_type='model'))
        def fetch(name):
            return Path(hf_hub_download(repo_id=repo,filename=name,token=token,
                        repo_type='model',local_dir=str(Path(workspace)/'phase1_download')))
        desc=repo
    weight=spec['weight']
    if weight not in files:
        # Do not silently take an unknown checkpoint and call it the final model.
        near=[x for x in files if x.endswith(('model.safetensors','model_state.pt'))]
        raise FileNotFoundError(f'Expected {weight} in {desc}. Available weight files: {near[:10]}')
    path=fetch(weight)
    if path.suffix=='.safetensors':state=load_file(str(path),device='cpu')
    else:state=torch.load(path,map_location='cpu',weights_only=True)
    for key in ('model','state_dict','model_state_dict','model_state'):
        if key in state and isinstance(state[key],dict):
            state=state[key]
            break
    state={k.removeprefix('module.').removeprefix('_orig_mod.'):v
           for k,v in state.items() if isinstance(v,torch.Tensor)}
    if 'token_embedding.weight' not in state:
        raise ValueError('Phase 1 weights do not contain token_embedding.weight; wrong model file')
    # Find tokenizer next to the model first, and then at repo root.
    relative_tokenizers=[str(prefix/'tokenizer.json'),'tokenizer.json']
    relative_tokenizers += [p for p in sorted(files) if p.endswith('/tokenizer.json')]
    relative_tokenizers=list(dict.fromkeys(relative_tokenizers))
    tokfile=next((p for p in relative_tokenizers if p in files),None)
    if not tokfile:raise FileNotFoundError('tokenizer.json is missing from Phase 1 repo')
    token_path=fetch(tokfile)
    tokenizer=PreTrainedTokenizerFast(tokenizer_file=str(token_path))
    vocab=state['token_embedding.weight'].shape[0]
    if len(tokenizer)!=vocab:
        raise ValueError(f'Tokenizer/weights mismatch: tokenizer={len(tokenizer)}, Phase 1 vocab={vocab}. No tokens were added.')
    # We only use pad ID for our batch mask. Do not add any token to the original vocabulary.
    toks=tokenizer.get_vocab()
    for field,choices in [('pad_token',('<pad>','[PAD]')),('eos_token',('</s>','<|endoftext|>'))]:
        for option in choices:
            if option in toks:
                setattr(tokenizer,field,option)
                break
    # A3/A4 attention settings are not recoverable from weight shapes alone.
    prefix_config=[str(prefix/'architecture_config.json'),'architecture_config.json']
    prefix_config += [p for p in sorted(files) if p.endswith('/architecture_config.json')]
    conf_file=next((p for p in dict.fromkeys(prefix_config) if p in files),None)
    cfg=json.loads(fetch(conf_file).read_text()) if conf_file else {}
    if arch in ('a3','a4') and not cfg:
        raise ValueError(f'{arch}: architecture_config.json is required to reproduce Phase 1 attention')
    print('Phase 1 repo:',desc)
    print('Weights:',weight)
    print('Tokenizer:',tokfile,'vocab:',vocab)
    print('Config:',conf_file or '(original source defaults)')
    return state,tokenizer,cfg,{'repo':desc,'weight':weight,'tokenizer':tokfile,'config':conf_file}


def download_phase7(api,token,workspace,local_folder=None):
    folder=Path(workspace)/'phase7_data';folder.mkdir(parents=True,exist_ok=True)
    result={}
    if local_folder:
        local=Path(local_folder)
        for key,relative in DATA_PATHS.items():
            p=local/relative
            if not p.exists():
                p=local/Path(relative).name
            if not p.exists():raise FileNotFoundError(relative)
            result[key]=p
    else:
        for key,relative in DATA_PATHS.items():
            result[key]=Path(hf_hub_download(repo_id=DATA_REPO,filename=relative,
                repo_type='dataset',token=token,local_dir=str(folder)))
    data={key:load_records(value,key) for key,value in result.items()}
    check_splits(data['train_q'],data['train_c'],data['dev_q'],data['dev_c'],
                 data['test_q'],data['test_c'])
    return data


def setup_model(arch,method,state,cfg,tokenizer,output,student_layers=None):
    base,dim,vocab,count=build_backbone(arch,state,cfg,student_layers)
    assert len(tokenizer)==vocab
    model=RetrievalModel(base,arch,dim,method)
    n_total=sum(p.numel() for p in model.parameters())
    n_train=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'{arch.upper()} / {method}: loaded {count} Phase 1 tensors')
    print(f'Params: {n_total/1e6:.2f}M total, {n_train/1e6:.2f}M trainable')
    save_json(Path(output)/'model_details.json',
        {'arch':arch,'method':method,'total_parameters':n_total,'trainable_parameters':n_train,
         'phase1_tensors_loaded':count,'tokenizer_vocab_size':len(tokenizer)})
    return model


def backup_to_private_hf(api,token,folder,repo_name):
    user=api.whoami(token=token)['name']
    target=f'{user}/{repo_name}'
    api.create_repo(repo_id=target,repo_type='model',private=True,exist_ok=True,token=token)
    info=api.repo_info(repo_id=target,repo_type='model',token=token)
    if not info.private:
        raise ValueError(f'The backup repo {target} is public. Use a private repo instead.')
    api.upload_folder(repo_id=target,repo_type='model',folder_path=str(folder),token=token,
                      commit_message='Save Phase 7 experiment files')
    print('Private HF backup:',target)
    return target


def restore_private_backup_if_available(api,token,folder,repo_name,method):
    # After Kaggle session restarts, try to recover last.pt from the private HF repo.
    user=api.whoami(token=token)['name']
    target=f'{user}/{repo_name}'
    try:
        info=api.repo_info(repo_id=target,repo_type='model',token=token)
    except Exception as exc:
        if ('404' in str(exc) or 'Repository Not Found' in str(exc)
                or 'not found' in str(exc).lower()):
            print('No earlier private backup found; this is a new run')
            return
        print('HF restore skipped:',str(exc)[:160])
        return
    if not info.private:
        raise ValueError('Cannot restore: the previous repo is public')
    files=set(api.list_repo_files(target,repo_type='model'))
    folders=['teacher','student'] if method=='kd' else ['train']
    restored=0
    for sub in folders:
        for name in ('last.pt','best.pt','training_log.json','dev_metrics.json'):
            relative=f'{sub}/{name}'
            local=Path(folder)/relative
            if relative in files and not local.is_file():
                local.parent.mkdir(parents=True,exist_ok=True)
                hf_hub_download(repo_id=target,repo_type='model',filename=relative,
                                local_dir=str(folder),token=token)
                restored+=1
    if restored:
        print(f'Restored {restored} previous training files from private {target}')


## 6. Log in and read my official files

In [ ]:
from huggingface_hub import HfApi
from phase7_backbones import RetrievalModel
from phase7_core import *
from phase7_setup import *

HF_TOKEN = get_hf_token()
api = HfApi(token=HF_TOKEN)
HF_USERNAME = api.whoami()['name']
print('HF account:', HF_USERNAME)

# Save the Python source with the other experiment outputs.
(OUT/'code').mkdir(exist_ok=True)
for name in ['phase7_backbones.py','phase7_core.py','phase7_setup.py']:
    shutil.copy2('/kaggle/working/'+name, OUT/'code'/name)

state, tokenizer, saved_cfg, source_info = download_phase1(
    ARCH, api, HF_TOKEN, '/kaggle/working/phase7_file_cache',
    local_folder=LOCAL_PHASE1_DIR or None)
data = download_phase7(api, HF_TOKEN, '/kaggle/working/phase7_file_cache',
    local_folder=LOCAL_PHASE7_DATA_DIR or None)
save_json(OUT/'phase1_used.json',source_info)
# If Kaggle discarded its local output, recover my last private checkpoint first.
restore_private_backup_if_available(api,HF_TOKEN,OUT,HF_BACKUP_REPO,METHOD)
print('Example query:',next(iter(data['train_q'])))
print('Example gold size:',len(gold_ids(next(iter(data['train_q'].values())))))


## 7. Build my retrieval model

In [ ]:
case_tokens = CaseTokens(tokenizer, max_tokens=MAX_TOKENS, n_chunks=CHUNKS_PER_CASE)
model = setup_model(ARCH,METHOD,state,saved_cfg,tokenizer,OUT)


## 8. Fine-tuning and keeping the best dev checkpoint

In [ ]:
SETTINGS = {'seed':SEED,'epochs':EPOCHS,'lr':LR,'accum_steps':ACCUM_STEPS,
            'save_steps':SAVE_STEPS,'embed_batch_size':EMBED_BATCH_SIZE,
            'hybrid_alpha':HYBRID_ALPHA,'max_train_queries':MAX_TRAIN_QUERIES}

# Backups are private and all Phase 7 folders have their own repo.
def backup_this_run():
    try:
        backup_to_private_hf(api,HF_TOKEN,OUT,HF_BACKUP_REPO)
    except Exception as exc:
        # Keep training. Last.pt and best.pt are saved locally before trying HF.
        print('HF backup warning:',str(exc)[:300])

model = train_model(model,data['train_q'],data['train_c'],
    data['dev_q'],data['dev_c'],case_tokens,DEVICE,OUT/'train',SETTINGS,
    backup_callback=backup_this_run if UPLOAD_EACH_EPOCH else None)


## 9. Dev score (local Micro-F1@7)

In [ ]:
# Check the best saved model again and write the dev numbers to my folder.
dev_predictions = rank_split(model,data['dev_q'],data['dev_c'],
    case_tokens,DEVICE,alpha=HYBRID_ALPHA,batch_size=EMBED_BATCH_SIZE)
dev_metrics = metric_at7(data['dev_q'],dev_predictions)
save_json(OUT/'dev_predictions_best.json',dev_predictions)
save_json(OUT/'dev_metrics_best.json',dev_metrics)
print('Dev metrics:',dev_metrics)


## 10. Test predictions and Codabench ZIP

In [ ]:
# Strict test candidate pool only. IDs stay as strings (including leading zeroes).
test_predictions = rank_split(model,data['test_q'],data['test_c'],
    case_tokens,DEVICE,alpha=HYBRID_ALPHA,batch_size=EMBED_BATCH_SIZE)
prediction_file, submission_zip = prepare_submission(
    test_predictions,data['test_q'],data['test_c'],OUT/'submission')
print('One prediction:',next(iter(test_predictions.items())))
print('Upload this ZIP to Codabench:',submission_zip)


## 11. Final private backup

In [ ]:
# This uploads the complete run folder: code, settings, metrics, checkpoints, JSON and ZIP.
backup_this_run()
print('Local output:', OUT)
print('Codabench file:', submission_zip)
